# Bloque I — Iniciación en análisis de datos con Python
## Notebook Entregable — Ejercicio Integrador

**Objetivos:**
- Configurar el entorno de trabajo
- Cargar, explorar y limpiar datos con pandas
- Construir un análisis descriptivo reproducible

> Pon el archivo `ventas_mayo2026.csv` en una carpeta `data/` al mismo nivel que este notebook.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DATA_DIR = Path('data')
pd.set_option('display.max_columns', 50)

df = pd.read_csv(DATA_DIR / 'ventas_mayo2026.csv', parse_dates=['fecha'])
print(f'Shape inicial: {df.shape}')
display(df.head())
df.info()

## Limpieza básica y tipos de datos

In [ ]:
# Eliminamos duplicados exactos
df_clean = df.drop_duplicates().copy()

# Imputamos nulos
df_clean['canal'] = df_clean['canal'].fillna('desconocido')
df_clean['importe'] = df_clean['importe'].fillna(df_clean['importe'].median())

# Columna derivada
df_clean['conversion'] = df_clean['clientes'] / df_clean['visitas']
df_clean['ticket_medio'] = df_clean['importe'] / df_clean['unidades']

print(f'Shape tras limpieza: {df_clean.shape}')
print(f'Nulos restantes:\n{df_clean.isnull().sum()}')
display(df_clean.describe(include='all'))

## Tarea 1 — Ventas totales por canal

In [ ]:
ventas_canal = (
    df_clean
    .groupby('canal', as_index=False)
    .agg(
        importe_total=('importe', 'sum'),
        importe_medio=('importe', 'mean'),
        operaciones=('importe', 'count'),
        unidades_total=('unidades', 'sum')
    )
    .sort_values('importe_total', ascending=False)
    .reset_index(drop=True)
)
ventas_canal['importe_total'] = ventas_canal['importe_total'].round(2)
ventas_canal['importe_medio'] = ventas_canal['importe_medio'].round(2)
display(ventas_canal)

## Tarea 2 — Ticket medio por región

In [ ]:
ticket_region = (
    df_clean
    .groupby('region', as_index=False)
    .agg(
        ticket_medio=('ticket_medio', 'mean'),
        importe_total=('importe', 'sum'),
        operaciones=('importe', 'count')
    )
    .sort_values('ticket_medio', ascending=False)
    .reset_index(drop=True)
)
ticket_region['ticket_medio'] = ticket_region['ticket_medio'].round(2)
ticket_region['importe_total'] = ticket_region['importe_total'].round(2)
display(ticket_region)

## Tarea 3 — Categoría con mayor unidades

In [ ]:
cat_resumen = (
    df_clean
    .groupby('categoria', as_index=False)
    .agg(
        unidades_total=('unidades', 'sum'),
        importe_total=('importe', 'sum'),
        ticket_medio=('ticket_medio', 'mean')
    )
    .sort_values('unidades_total', ascending=False)
    .reset_index(drop=True)
)
cat_resumen['importe_total'] = cat_resumen['importe_total'].round(2)
cat_resumen['ticket_medio'] = cat_resumen['ticket_medio'].round(2)
display(cat_resumen)

top_cat = cat_resumen.iloc[0]['categoria']
print(f'Categoría con más unidades vendidas: {top_cat}')

## Tarea 4 — Gráficos por canal y región

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores = ['#378ADD','#1D9E75','#D85A30','#D4537E','#888780']

# Gráfico 1: Ventas totales por canal
ax1 = axes[0]
bars1 = ax1.bar(ventas_canal['canal'], ventas_canal['importe_total'],
                color=colores[:len(ventas_canal)], edgecolor='white', linewidth=0.5)
ax1.set_title('Ventas totales por canal', fontsize=13, pad=10)
ax1.set_xlabel('Canal')
ax1.set_ylabel('Importe total (€)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))
for bar in bars1:
    h = bar.get_height()
    ax1.text(bar.get_x()+bar.get_width()/2, h*1.01,
             f'{h/1e6:.2f}M', ha='center', va='bottom', fontsize=9)
ax1.spines[['top','right']].set_visible(False)

# Gráfico 2: Operaciones por canal
ax2 = axes[1]
bars2 = ax2.bar(ventas_canal['canal'], ventas_canal['operaciones'],
                color=colores[:len(ventas_canal)], edgecolor='white', linewidth=0.5)
ax2.set_title('Número de operaciones por canal', fontsize=13, pad=10)
ax2.set_xlabel('Canal')
ax2.set_ylabel('Operaciones')
for bar in bars2:
    h = bar.get_height()
    ax2.text(bar.get_x()+bar.get_width()/2, h*1.01,
             f'{int(h):,}', ha='center', va='bottom', fontsize=9)
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_canal.png', dpi=150, bbox_inches='tight')
plt.show()

# Gráfico horizontal: ticket medio por región
fig2, ax3 = plt.subplots(figsize=(8, 4))
ticket_sorted = ticket_region.sort_values('ticket_medio')
bars3 = ax3.barh(ticket_sorted['region'], ticket_sorted['ticket_medio'],
                 color='#378ADD', edgecolor='white')
ax3.set_title('Ticket medio por región (€)', fontsize=13, pad=10)
ax3.set_xlabel('Ticket medio (€)')
for bar in bars3:
    w = bar.get_width()
    ax3.text(w+0.5, bar.get_y()+bar.get_height()/2,
             f'{w:.2f}€', va='center', fontsize=9)
ax3.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('grafico_region.png', dpi=150, bbox_inches='tight')
plt.show()

## Tarea 5 — Conclusiones de negocio

In [ ]:
# Conclusiones generadas dinámicamente a partir de los resultados

mejor_canal = ventas_canal.iloc[0]['canal']
mejor_canal_importe = ventas_canal.iloc[0]['importe_total']
mejor_region_ticket = ticket_region.iloc[0]['region']
mejor_region_val = ticket_region.iloc[0]['ticket_medio']
top_categoria = cat_resumen.iloc[0]['categoria']
top_cat_unidades = cat_resumen.iloc[0]['unidades_total']
canal_conv = df_clean.groupby('canal')['conversion'].mean().sort_values(ascending=False)
mejor_conv_canal = canal_conv.index[0]
mejor_conv_val = canal_conv.iloc[0]

conclusiones = [
    f'1. El canal **{mejor_canal}** lidera en volumen de ventas con '
    f'{mejor_canal_importe:,.0f}€ acumulados en mayo 2026. '
    f'Concentrar inversión en este canal o replicar sus dinámicas en otros '
    f'podría tener un impacto directo en la facturación.',

    f'2. La región **{mejor_region_ticket}** presenta el ticket medio más alto '
    f'({mejor_region_val:.2f}€/unidad), lo que sugiere una mayor disposición a pagar '
    f'o un mix de producto más premium. Es candidata a estrategias de upselling.',

    f'3. La categoría **{top_categoria}** es la que más unidades mueve '
    f'({top_cat_unidades:,} uds.), pero hay que contrastar su ticket medio: '
    f'alto volumen no siempre implica mayor rentabilidad. '
    f'Además, el canal **{mejor_conv_canal}** registra la mejor tasa de conversión '
    f'({mejor_conv_val:.1%}), lo que no coincide necesariamente con el de mayor volumen — '
    f'señal de que eficiencia y alcance son palancas distintas a gestionar por separado.',
]

print('=== CONCLUSIONES DE NEGOCIO ===')
for c in conclusiones:
    # limpiamos el markdown para el print
    print(c.replace('**','') + '\n')

---
*Notebook ejecutado — Bloque I · Mayo 2026*